# 01 — Create the same agent seven ways

One agent, the support-ticket assistant, built every way Foundry allows. The point is not the agent. The point is to see where the **definition** lives and where the **loop** runs in each case, because that decides what you can version, govern, trace and roll back. Every variant is then called through the same surface, `POST /responses`.

| # | Way | Definition lives | Loop runs | Status | File |
|---|---|---|---|---|---|
| 1 | Prompt agent, portal | Foundry, versioned | Foundry | GA | `01-prompt-agent-portal.md` |
| 2 | Prompt agent, SDK / REST / azd | Foundry, versioned | Foundry | GA | `02-prompt-agent-sdk.py`, `02b-prompt-agent-azd/` |
| 2b | Same agent driven from Agent Framework (`FoundryAgent`), and exported from Agent Framework (`to_prompt_agent`) | Foundry | Foundry | GA | `05-foundry-agent-af.py` |
| 3 | Voice prompt agent | Foundry, `kind: voice` | Foundry + Voice Live | preview | `03-voice-prompt-agent.py` |
| 4 | Ephemeral agent (Responses API) | your request | Foundry runtime, nothing to version | GA | `04-ephemeral-agent.py` |
| 5 | Hosted agent from source zip | your container | Foundry sandbox | GA | `../02-hosted-agents/code-deploy/` |
| 6 | Hosted agent from container image | your container | Foundry sandbox | GA | `../02-hosted-agents/container-deploy/` |
| 7 | Self-hosted on your compute, Foundry models + toolbox | your infra | your infra | GA | `../02-hosted-agents/self-hosted-aca/` |

Then `06-invoke-everything.py` calls whichever of them exist, identically, and `07-versions-and-publish.py` shows versions, routing and what "publish" now means.

Decision rule to carry: *declarative loop → prompt agent; custom loop, dependencies or state → hosted; nothing to manage → ephemeral; your own platform constraints → self-hosted.*



In [ ]:
# Shared helpers (same as _common.py)
from __future__ import annotations

import os
from pathlib import Path

from azure.ai.projects import AIProjectClient
from azure.core.exceptions import ResourceNotFoundError
from azure.identity import DefaultAzureCredential
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))  # .env at the track root (walks up)

PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"].rstrip("/")
MODEL = os.environ.get("FOUNDRY_MODEL", "gpt-5-mini")

SUPPORT_INSTRUCTIONS = (
    "You are the support-ticket assistant for a UK bank's card team. "
    "Answer briefly. When you do not have a tool for something, say so instead of guessing."
)


def project_client(**kwargs) -> AIProjectClient:
    """One client per script. `allow_preview=True` is only needed for preview surfaces (voice)."""
    return AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=DefaultAzureCredential(), **kwargs)


def get_or_create(project: AIProjectClient, agent_name: str, definition):
    """Return the existing agent if it exists, else create version 1.

    The docs say to treat 404 as terminal: `get` raising ResourceNotFoundError means
    the agent does not exist, so we create it. Anything else is a real error.
    Every `create_version` call adds an immutable version (cap 1,000 per agent), so
    scripts must not call it unconditionally on each run.
    """
    try:
        agent = project.agents.get(agent_name=agent_name)
        print(f"[get-or-create] reusing '{agent.name}' (latest version {agent.versions.latest.version})")
        return agent
    except ResourceNotFoundError:
        version = project.agents.create_version(agent_name=agent_name, definition=definition)
        print(f"[get-or-create] created '{version.name}' version {version.version}")
        return project.agents.get(agent_name=agent_name)


def delete_agent(project: AIProjectClient, agent_name: str) -> None:
    try:
        project.agents.delete(agent_name=agent_name)
        print(f"[cleanup] deleted '{agent_name}' and all its versions")
    except ResourceNotFoundError:
        pass


# Way 1 — Prompt agent in the portal

The portal is the fastest path and the one most teams start with. It is also the one that hides the most, so this page shows what each click does on the wire.

1. Foundry portal → your project → **Build** → **Agents** → **Create agent**. Interaction mode **Text** (Voice is way 3 and cannot be changed later).
2. Name `ticket-agent-portal`, model `gpt-5-mini`, instructions:
   > You are the support-ticket assistant for a UK bank's card team. Answer briefly. When you do not have a tool for something, say so instead of guessing.
3. **Save**. Saving creates **version 1**. Every later Save creates a new immutable version; the name can never change.
4. **Try in playground** → ask "My card was charged twice." The playground is calling `POST {project_endpoint}/agents/ticket-agent-portal/endpoint/protocols/openai/responses?api-version=v1`.
5. **Code** tab → the SDK equivalent is exactly `02-prompt-agent-sdk.py`.

What the Save button sent (REST, `api-version=v1`):
```http
POST https://<resource>.services.ai.azure.com/api/projects/<project>/agents?api-version=v1
Authorization: Bearer <token for https://ai.azure.com/.default>
{"name": "ticket-agent-portal", "definition": {"kind": "prompt", "model": "gpt-5-mini", "instructions": "..."}}
```
And what the playground sends per turn:
```http
POST .../agents/ticket-agent-portal/endpoint/protocols/openai/responses?api-version=v1
{"input": "My card was charged twice.", "conversation": "<conversation id>"}
```

Proof: screenshot of the agent's **Versions** tab showing v1, and the playground answer. Then run `python 06-invoke-everything.py` with `AGENT_NAMES=ticket-agent-portal` to prove the same agent answers from code.

Gotcha: the portal's default "create a new version on every save" is fine; the same pattern in a startup hook of an app is how teams hit the 1,000-version cap.


## `02-prompt-agent-sdk.py`

Way 2 — Prompt agent created with the Projects SDK, invoked through the Responses API.

The definition (model, instructions, tools) lives in Foundry as an immutable version.
Foundry runs the loop. You call it through the one invocation surface: POST /responses.

Run:  python 02-prompt-agent-sdk.py [--keep]
Proof: prints the created version, a two-turn conversation on the server side, the
       version list, and (unless --keep) deletes the agent.

In [ ]:
import sys
sys.argv=['nb','--keep'] if 'keep' in locals() else sys.argv  # notebooks keep agents by default

from azure.ai.projects.models import PromptAgentDefinition

# from _common import MODEL, SUPPORT_INSTRUCTIONS, delete_agent, get_or_create, project_client

AGENT_NAME = "ticket-agent-prompt"

project = project_client()

# 1. Definition stored in Foundry. Tools are added in F04; here it is model + instructions.
definition = PromptAgentDefinition(model=MODEL, instructions=SUPPORT_INSTRUCTIONS)
agent = get_or_create(project, AGENT_NAME, definition)

# 2. Invoke. get_openai_client(agent_name=...) returns an OpenAI client pre-bound to the
#    agent endpoint, so you do not pass model/instructions and you do not need extra_body.
openai = project.get_openai_client(agent_name=AGENT_NAME)
conversation = openai.conversations.create()            # server-side state
r1 = openai.responses.create(conversation=conversation.id, input="My card was charged twice for order 8841.")
print("turn 1:", r1.output_text)
r2 = openai.responses.create(conversation=conversation.id, input="What did I just tell you the order number was?")
print("turn 2:", r2.output_text)                          # proves the conversation carried state

# 3. Same thing without a conversation object: chain by previous_response_id.
r3 = openai.responses.create(input="Say 'ready' and nothing else.")
r4 = openai.responses.create(input="Repeat what you just said.", previous_response_id=r3.id)
print("chained:", r4.output_text)

# 4. Versions are immutable and listable. Refer to one as name:version when you must pin.
versions = list(project.agents.list_versions(agent_name=AGENT_NAME))
print("versions:", [v.version for v in versions])

# 5. Cleanup unless asked to keep it (06-invoke-everything.py and 07 reuse it).
if "--keep" not in sys.argv:
    delete_agent(project, AGENT_NAME)

## `05-foundry-agent-af.py`

Way 2b — A server-managed prompt agent driven from Agent Framework with `FoundryAgent`.

`FoundryChatClient` = your code owns the definition (way 4).
`FoundryAgent`      = the definition lives in Foundry (a prompt or hosted agent); Agent
Framework is only the client. This replaces the removed `AzureAIAgentClient`.

It also shows `to_prompt_agent()`: build the agent locally with Agent Framework, then
export the definition to Foundry so the same agent exists as a versioned prompt agent.

Run:  python 05-foundry-agent-af.py   (creates ticket-agent-prompt if 02 did not keep it)

In [ ]:
import asyncio

from agent_framework import Agent
from agent_framework.foundry import FoundryAgent, FoundryChatClient, to_prompt_agent
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import DefaultAzureCredential

# from _common import MODEL, PROJECT_ENDPOINT, SUPPORT_INSTRUCTIONS, delete_agent, get_or_create, project_client

AGENT_NAME = "ticket-agent-prompt"
EXPORTED_NAME = "ticket-agent-exported"


async def main() -> None:
    project = project_client()
    get_or_create(project, AGENT_NAME, PromptAgentDefinition(model=MODEL, instructions=SUPPORT_INSTRUCTIONS))

    # 1. Drive the Foundry-managed agent from Agent Framework. No instructions here: Foundry has them.
    agent = FoundryAgent(project_endpoint=PROJECT_ENDPOINT, agent_name=AGENT_NAME, credential=DefaultAzureCredential())
    result = await agent.run("Explain in one line what you are.")
    print("[FoundryAgent]", result.text)

    # 2. Build locally, export to Foundry. to_prompt_agent() turns an Agent Framework Agent
    #    into a PromptAgentDefinition you can create_version with.
    local = Agent(
        client=FoundryChatClient(project_endpoint=PROJECT_ENDPOINT, model=MODEL, credential=DefaultAzureCredential()),
        instructions=SUPPORT_INSTRUCTIONS + " You were authored in Agent Framework and exported.",
        name=EXPORTED_NAME,
    )
    definition = to_prompt_agent(local)
    print("[to_prompt_agent] exported definition model:", definition.model)
    get_or_create(project, EXPORTED_NAME, definition)
    exported = project.get_openai_client(agent_name=EXPORTED_NAME).responses.create(input="Where were you authored?")
    print("[exported agent]", exported.output_text)
    delete_agent(project, EXPORTED_NAME)


if __name__ == "__main__":
    asyncio.run(main())

## `04-ephemeral-agent.py`

Way 4 — Ephemeral agent: the definition lives in your code, per request. No agent resource.

Two spellings of the same thing:
  (a) Agent Framework `FoundryChatClient` — what hosted agents use under the hood.
  (b) A plain OpenAI client pointed at {project_endpoint}/openai/v1 — proves there is
      nothing Foundry-specific in the wire format.
Both still get Foundry's runtime (conversations, hosted tools, tracing, identity).
Nothing is created in Foundry, so there is nothing to version, publish, or clean up.

Run:  python 04-ephemeral-agent.py

In [ ]:
import asyncio
import os

from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

# from _common import MODEL, PROJECT_ENDPOINT, SUPPORT_INSTRUCTIONS

TICKETS = {"8841": {"status": "open", "issue": "duplicate charge", "amount_gbp": 42.10}}


@tool(approval_mode="never_require")
def get_ticket(ticket_id: str) -> dict:
    """Look up a support ticket by id."""
    return TICKETS.get(ticket_id, {"error": "not found"})


async def with_agent_framework() -> None:
    client = FoundryChatClient(project_endpoint=PROJECT_ENDPOINT, model=MODEL, credential=DefaultAzureCredential())
    agent = Agent(client=client, instructions=SUPPORT_INSTRUCTIONS, tools=[get_ticket], name="ticket-agent-ephemeral")
    result = await agent.run("What is the status of ticket 8841 and how much is involved?")
    print("[agent-framework]", result.text)

    print("[agent-framework, streaming] ", end="", flush=True)
    async for chunk in agent.run("Summarise the ticket in five words.", stream=True):
        if chunk.text:
            print(chunk.text, end="", flush=True)
    print()

    # Reasoning-effort variant on the same client (replaces the old 02-af-standard-agent-resoning.py).
    reasoning_model = os.environ.get("FOUNDRY_REASONING_MODEL", MODEL)
    reasoning_agent = Agent(
        client=FoundryChatClient(project_endpoint=PROJECT_ENDPOINT, model=reasoning_model, credential=DefaultAzureCredential()),
        instructions="Think carefully, answer in one sentence.",
        default_options={"reasoning": {"effort": "low"}},
    )
    r = await reasoning_agent.run("A customer disputes a £42.10 charge made twice. What is the refund amount?")
    print("[agent-framework, reasoning]", r.text)


def with_raw_openai_client() -> None:
    token = get_bearer_token_provider(DefaultAzureCredential(), "https://ai.azure.com/.default")
    oai = OpenAI(base_url=f"{PROJECT_ENDPOINT}/openai/v1", api_key=token)
    r = oai.responses.create(model=MODEL, instructions=SUPPORT_INSTRUCTIONS, input="Who are you and what can you do?")
    print("[raw openai client]", r.output_text)


if __name__ == "__main__":
    asyncio.run(with_agent_framework())
    with_raw_openai_client()

## `03-voice-prompt-agent.py`

Way 3 — Voice prompt agent (preview). Same create_version surface, kind=voice.

A voice agent is a prompt agent whose definition carries the realtime model and audio
config. Its endpoint is live when version 1 exists. You talk to it over a realtime
WebSocket (Voice Live) rather than POST /responses. `azd ai agent invoke` cannot do voice.

Requires: a `gpt-realtime` deployment (FOUNDRY_VOICE_MODEL), access to the voice preview,
and `pip install "azure-ai-projects[voice]"` (already in requirements via extras? no: run it once).

Run:  python 03-voice-prompt-agent.py
Proof: version created, a text-in / text-out realtime turn, conversation listed, agent deleted.

In [ ]:
import os

from azure.ai.projects.models import (
    AgentKind,
    VoiceAgentAudioConfig,
    VoiceAgentAudioOutputConfig,
    VoiceAgentDefinition,
    VoiceModelType,
    VoiceOutputModality,
    VoiceType,
)

# from _common import SUPPORT_INSTRUCTIONS, delete_agent, get_or_create, project_client

AGENT_NAME = "ticket-agent-voice"
VOICE_MODEL = os.environ.get("FOUNDRY_VOICE_MODEL", "gpt-realtime")

# allow_preview=True is required for voice definitions through .agents
project = project_client(allow_preview=True)

definition = VoiceAgentDefinition(
    model_type=VoiceModelType.MANAGED,          # service-hosted realtime model; SELF_DEPLOYED = your deployment name
    model=VOICE_MODEL,
    instructions=SUPPORT_INSTRUCTIONS + " Keep spoken replies under two sentences.",
    audio=VoiceAgentAudioConfig(
        output=VoiceAgentAudioOutputConfig(voice="en-GB-SoniaNeural", voice_type=VoiceType.AZURE_STANDARD)
    ),
    output_modalities=[VoiceOutputModality.AUDIO],
    store=True,                                  # keep transcript + audio (60-day retention)
)
agent = get_or_create(project, AGENT_NAME, definition)
print("voice agents in project:", [a.name for a in project.agents.list(kind=AgentKind.VOICE)])

# Realtime session: text in, read the transcript events back. Audio is PCM16 mono 24 kHz.
with project.beta.voice_agents.realtime.connect(agent_name=AGENT_NAME) as conn:
    first = conn.recv(timeout=45)
    conversation_id = getattr(getattr(first, "session", None), "conversation_id", None) or getattr(first, "conversation_id", None)
    print("session event:", first.type, "conversation:", conversation_id)
    conn.conversation.item.create(
        item={"type": "message", "role": "user", "content": [{"type": "input_text", "text": "My card was charged twice. What should I do?"}]}
    )
    conn.response.create()
    transcript = []
    while True:
        event = conn.recv(timeout=45)
        if event.type == "response.output_audio_transcript.delta":
            transcript.append(event.delta)
        elif event.type in ("response.done", "error"):
            break
    print("agent said:", "".join(transcript) or f"(no transcript; last event {event.type})")

if conversation_id:
    items = list(project.beta.voice_agents.conversations.list_items(AGENT_NAME, conversation_id, order="asc"))
    print(f"stored conversation {conversation_id}: {len(items)} item(s)")

delete_agent(project, AGENT_NAME)

## `06-invoke-everything.py`

One invocation surface. Call every agent in this repo the same way and show the four
Responses-API controls that matter in production: conversation, previous_response_id,
stream, background.

Agents are looked up by name; those that do not exist are skipped with a note, so this
script is a progress meter for the workshop. Names it looks for:
  ticket-agent-prompt        (01/02, prompt)         ticket-agent-hosted-code  (02/code-deploy)
  ticket-agent-hosted        (02/container-deploy)   ticket-agent-workflow     (12)
  AGENT_NAMES env var overrides the list (comma separated).

Run:  python 06-invoke-everything.py

In [ ]:
import os
import time

from azure.core.exceptions import ResourceNotFoundError

# from _common import project_client

DEFAULT_NAMES = ["ticket-agent-prompt", "ticket-agent-hosted-code", "ticket-agent-hosted", "ticket-agent-workflow"]
names = [n.strip() for n in os.environ.get("AGENT_NAMES", ",".join(DEFAULT_NAMES)).split(",") if n.strip()]
project = project_client()

for name in names:
    try:
        details = project.agents.get(agent_name=name)
    except ResourceNotFoundError:
        print(f"\n== {name}: not deployed yet, skipping")
        continue
    kind = getattr(details.versions.latest.definition, "kind", "?")
    print(f"\n== {name} (kind={kind}, latest v{details.versions.latest.version})")
    openai = project.get_openai_client(agent_name=name)

    # 1. conversation: server-side state, works identically for prompt and hosted agents
    conv = openai.conversations.create()
    r1 = openai.responses.create(conversation=conv.id, input="Remember the code word 'heron'. Reply OK.")
    r2 = openai.responses.create(conversation=conv.id, input="What is the code word?")
    print("  conversation:", r2.output_text.strip()[:80])

    # 2. previous_response_id: client-side chaining, no conversation object
    r3 = openai.responses.create(input="Reply with the single word 'alpha'.")
    r4 = openai.responses.create(input="Now the Greek letter after it.", previous_response_id=r3.id)
    print("  chained:     ", r4.output_text.strip()[:80])

    # 3. stream: SSE deltas, same for every agent type
    print("  stream:       ", end="", flush=True)
    for event in openai.responses.create(input="Count to five, digits only.", stream=True):
        if event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)
    print()

    # 4. background: submit, poll status. Needed when a run can exceed the request timeout.
    bg = openai.responses.create(input="Write one sentence about herons.", background=True)
    while bg.status in ("queued", "in_progress"):
        time.sleep(1)
        bg = openai.responses.retrieve(bg.id)
    print("  background:  ", bg.status, "-", (bg.output_text or "").strip()[:80])

## `07-versions-and-publish.py`

Versions, the agent endpoint, and what "publish" means now.

- Every create_version is immutable. The endpoint serves ONE version at 100 % traffic;
  which one is chosen by the version selector (default: latest). No traffic splitting.
- Creating the agent already gave it a stable endpoint and its own Entra identity. The
  old "Agent Applications" publishing path is legacy. "Publish" now means distributing
  to Microsoft 365 Copilot / Teams (publish_to_microsoft365), which needs Bot Service rights.

Run:  python 07-versions-and-publish.py
Proof: two versions exist; the endpoint is pinned to v1 then back to latest; the M365
       publish defaults are printed (publishing itself is left as a commented call).

In [ ]:
from azure.ai.projects.models import (
    AgentEndpointConfig,
    FixedRatioVersionSelectionRule,
    PromptAgentDefinition,
    VersionSelector,
)

# from _common import MODEL, SUPPORT_INSTRUCTIONS, delete_agent, get_or_create, project_client

AGENT_NAME = "ticket-agent-prompt"
project = project_client()
get_or_create(project, AGENT_NAME, PromptAgentDefinition(model=MODEL, instructions=SUPPORT_INSTRUCTIONS))

# A second version with a visible behavioural change.
v2 = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(model=MODEL, instructions=SUPPORT_INSTRUCTIONS + " Always end with 'v2'."),
    description="adds the v2 marker",
)
print("created version", v2.version)
versions = sorted(int(v.version) for v in project.agents.list_versions(agent_name=AGENT_NAME))
print("versions:", versions)

openai = project.get_openai_client(agent_name=AGENT_NAME)
print("latest answers:", openai.responses.create(input="Say hello.").output_text.strip()[:60])

# Pin the endpoint to version 1 (rollback), then back to the newest. 100 % is the only ratio allowed.
def route_to(version: int) -> None:
    project.agents.update_details(
        agent_name=AGENT_NAME,
        agent_endpoint=AgentEndpointConfig(
            version_selector=VersionSelector(
                version_selection_rules=[FixedRatioVersionSelectionRule(agent_version=str(version), traffic_percentage=100)]
            )
        ),
    )
    print(f"endpoint now routes 100% to v{version}")

route_to(versions[0])
print("after rollback:", openai.responses.create(input="Say hello.").output_text.strip()[:60])
route_to(versions[-1])
print("after roll-forward:", openai.responses.create(input="Say hello.").output_text.strip()[:60])

# Identity and endpoint come with the agent; nothing to "publish" for a stable URL.
details = project.agents.get(agent_name=AGENT_NAME)
print("agent endpoint:", getattr(details, "agent_endpoint", None))
print("identity:", getattr(details, "identity", None) or getattr(details, "instance_identity", None))

# Distribution to Copilot / Teams. Needs Microsoft.BotService/* write rights; left as a dry run.
defaults = project.agents.get_microsoft365_publish_defaults(agent_name=AGENT_NAME)
print("m365 publish defaults:", defaults)
# project.agents.publish_to_microsoft365(agent_name=AGENT_NAME, publish_scope="Shared", agent_display_name="Ticket agent")

delete_agent(project, AGENT_NAME)